# Checkpoint 3 — Mock Contest 3 Solutions

Reference solutions for all 7 problems. Each is a complete stdin/stdout program; the cells below are display-only mirrors of the solver files in the `assets/` folder.

## Question 1

**Notice:** recursive backtracking count: place each unused value whose absolute difference from the previous placed value is not 1, marking/unmarking `used[]`; print the number of full arrangements.

In [ ]:
import sys

data = sys.stdin.read()
parts = data.split()
value_count = int(parts[0])
values = []
used = []
i = 0
while i < value_count:
    values.append(int(parts[i + 1]))
    used.append(False)
    i = i + 1

def count_arrangements(depth, previous_index):
    if depth == value_count:
        return 1
    count = 0
    i = 0
    while i < value_count:
        allowed = previous_index == -1
        if previous_index != -1:
            allowed = abs(values[i] - values[previous_index]) != 1
        if used[i] == False and allowed:
            used[i] = True
            count = count + count_arrangements(depth + 1, i)
            used[i] = False
        i = i + 1
    return count

print(str(count_arrangements(0, -1)))


## Question 2

**Notice:** push each bit with `appendleft`. For `NOT`, `popleft` one bit and push `1 - bit`. For `AND`, `OR` and `IMP`, `popleft` the RIGHT operand first and the LEFT operand second, then push the gate's result, where `left IMP right` is `0` only for `1 IMP 0`. Every token does a constant amount of stack work, so 200000 tokens take one quick pass; the one bit left on the stack is the answer.

In [ ]:
from collections import deque
import sys

data = sys.stdin.read()
tokens = data.split()
stack = deque()
i = 0
while i < len(tokens):
    token = tokens[i]
    if token == "NOT":
        value = stack.popleft()
        stack.appendleft(1 - value)
    elif token == "AND" or token == "OR" or token == "IMP":
        right = stack.popleft()
        left = stack.popleft()
        if token == "AND":
            if left == 1 and right == 1:
                value = 1
            else:
                value = 0
        elif token == "OR":
            if left == 1 or right == 1:
                value = 1
            else:
                value = 0
        else:
            if left == 1 and right == 0:
                value = 0
            else:
                value = 1
        stack.appendleft(value)
    else:
        stack.appendleft(int(token))
    i = i + 1
print(str(stack.popleft()))


## Question 3

**Notice:** repeatedly take `number % base` as the next digit (from the right) and divide with `number // base`. Prepend each digit to `forward` so the string reads in the correct order, and append it to `backward` to get the digits in reverse. Handle `0` separately as `"0"`. The representation is a palindrome exactly when `forward == backward`. Each step divides by at least 2, so even `10^18` needs only about 60 steps.

In [ ]:
import sys

data = sys.stdin.read()
parts = data.split()
number = int(parts[0])
base = int(parts[1])
forward = ""
backward = ""
if number == 0:
    forward = "0"
    backward = "0"
while number > 0:
    digit = str(number % base)
    forward = digit + forward
    backward = backward + digit
    number = number // base
if forward == backward:
    answer = "YES"
else:
    answer = "NO"
print(forward)
print(answer)


## Question 4

**Notice:** enumerate all `2^n` subsets with a bitmask and count those whose selected weights sum to the target.

In [ ]:
import sys

data = sys.stdin.read()
parts = data.split()
item_count = int(parts[0])
target = int(parts[1])
weights = []
i = 0
while i < item_count:
    weights.append(int(parts[i + 2]))
    i = i + 1

matching = 0
for mask in range(1 << item_count):
    total = 0
    i = 0
    while i < item_count:
        if mask & (1 << i):
            total = total + weights[i]
        i = i + 1
    if total == target:
        matching = matching + 1
print(str(matching))


## Question 5

**Notice:** fold an iterative Euclidean `gcd` across the values for the overall GCD, and build the LCM with `multiple // pair_gcd * value`; print `gcd lcm`.

In [ ]:
import sys

data = sys.stdin.read()
parts = data.split()
value_count = int(parts[0])

def gcd(a, b):
    while b != 0:
        a, b = b, a % b
    return a

common = int(parts[1])
multiple = int(parts[1])
i = 1
while i < value_count:
    value = int(parts[i + 1])
    common = gcd(common, value)
    pair_gcd = gcd(multiple, value)
    multiple = multiple // pair_gcd * value
    i = i + 1
result = str(common)
result = result + " "
result = result + str(multiple)
print(result)


## Question 6

**Notice:** run one Sieve of Eratosthenes through `N`, crossing off multiples of each prime starting at `prime * prime`. Starting at the square is safe because smaller multiples were already crossed off by smaller primes, and it must include the square itself (`169` is not prime). Then count every `low` with `low + 2 <= N` where both `low` and `low + 2` are still marked prime. The sieve does about `N log log N` work plus one linear pass, far faster than trial division on every candidate.

In [ ]:
import sys

data = sys.stdin.read()
limit = int(data.strip())
is_prime = []
i = 0
while i <= limit:
    is_prime.append(True)
    i = i + 1
is_prime[0] = False
is_prime[1] = False

prime = 2
while prime * prime <= limit:
    if is_prime[prime]:
        multiple = prime * prime
        while multiple <= limit:
            is_prime[multiple] = False
            multiple = multiple + prime
    prime = prime + 1

pairs = 0
low = 2
while low + 2 <= limit:
    if is_prime[low] and is_prime[low + 2]:
        pairs = pairs + 1
    low = low + 1
print(str(pairs))


## Question 7

**Notice:** `heaviest(node)` gives back the best root-to-leaf total starting at `node`. A leaf gives back its own value. A node with one child adds its value to that child's result, because a missing child is NOT a path worth `0`, and a path may not stop at a non-leaf. A node with two children adds its value to the larger of the two results. Each node is visited once, so the work is linear in `N`, and the height limit of 900 keeps the recursion under Python's default depth limit.

In [ ]:
import sys

data = sys.stdin.read()
parts = data.split()
node_count = int(parts[0])
root = int(parts[1])
values = []
left = []
right = []
i = 0
while i < node_count:
    values.append(int(parts[i * 3 + 2]))
    left.append(int(parts[i * 3 + 3]))
    right.append(int(parts[i * 3 + 4]))
    i = i + 1

def heaviest(node):
    if left[node] == -1 and right[node] == -1:
        return values[node]
    if left[node] == -1:
        return values[node] + heaviest(right[node])
    if right[node] == -1:
        return values[node] + heaviest(left[node])
    return values[node] + max(heaviest(left[node]), heaviest(right[node]))

print(str(heaviest(root)))
